# Soft Robotic Arm — Pouch Control and Reference Tracking

This notebook teaches you how to:

1. command all twenty pouches independently;
2. read the five pouch pressures in every segment;
3. observe the arm tip position and velocity;
4. use a PD controller to track a smooth random tip trajectory; and
5. render the controller's motion as a simulation video.

The final example is a starting point: tune the gains and pouch distribution,
then improve or replace the controller.


## 1 — Install the arm library

Run this once in a new Google Colab runtime. It installs [soft-robotic-arm](https://pypi.org/project/soft-robotic-arm/), a
ready-to-use MuJoCo simulation of the four-segment pneumatic arm. The package
includes the configured robot model and its calibration data, so you do not
need to download the repository, open the model files, or build the robot
yourself. MuJoCo, NumPy, Matplotlib, and the video tools are installed with it.

After installation, you use the package through a small public interface:

```python
from soft_robotic_arm import make_sim

sim = make_sim(control_hz=100, seed=0)
obs = sim.reset()
obs = sim.step(command)
```

- `make_sim(...)` creates a correctly configured arm simulation.
- `reset()` returns the arm to its initial state and gives you its first sensor reading.
- `step(command)` applies one pressure command and advances the simulation by one
  control interval.
- `command` may contain four segment pressures with shape `(4,)`, or twenty
  individual pouch pressures with shape `(4, 5)`.
- The returned `obs` dictionary contains the tip position and velocity, simulation
  time, and measured segment and pouch pressures.

The install command downloads the published package and all required dependencies
directly from PyPI. If an older version was already loaded in Colab, the setup
cell reloads the installed package before creating the simulator.


In [ ]:
%pip install -q --upgrade --no-cache-dir soft-robotic-arm


## 2 — Segment and pouch indexing

A pouch command has shape `(4, 5)`. Rows select S1–S4 and columns select
P1–P5. For example, `command[1, 2]` addresses P3 in S2.

<img src="https://raw.githubusercontent.com/Jeevan-HM/Soft-Robotic-Arm/simulation/mujoco/coursework/assets/soft_arm_hardware_and_layout.png" alt="Soft robotic arm hardware beside its segment and pouch indexing diagram" width="100%"/>

*Physical arm and indexing diagram. P1 is the bottom pouch and P5 is the top pouch in each segment.*

| Row | Segment | Tip tendency |
|---:|---|---|
| `0` | S1 | +x |
| `1` | S2 | +y |
| `2` | S3 | −x |
| `3` | S4 | −y |


In [ ]:
import os
import sys
from importlib.metadata import version

# Google Colab is Linux. Set the renderer before importing MuJoCo.
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# Colab can retain an already-imported package after %pip upgrades it.
# Remove only this package from the import cache, then load the installed copy.
for module_name in tuple(sys.modules):
    if module_name == "soft_robotic_arm" or module_name.startswith("soft_robotic_arm."):
        del sys.modules[module_name]

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import soft_robotic_arm
from IPython.display import Video, display
from soft_robotic_arm import make_sim

# CHANGED: added make_calibrated_sim -- the real/graded plant. It has the
# same arm model as make_sim(), plus a 0.5s transport delay and per-segment
# regulator gain/bias miscalibration fitted from real hardware runs. Kept
# make_sim() too so you can still compare against the easier classroom plant.
from soft_robotic_arm import make_calibrated_sim

# CHANGED: added -- this is the real evaluation harness (not written by us,
# it ships inside the installed soft_robotic_arm package). CircularTrackingTask
# defines the graded task (7mm circle, 0.1Hz, settle+track timing, baseline
# psi). evaluate_controller() runs that task end-to-end and returns the
# metrics (rmse_mm, max_error_mm, phase_lag_s, effort_psi) used in Cell 12.
from soft_robotic_arm.evaluation import evaluate_controller, CircularTrackingTask, make_reference

installed_version = version("soft-robotic-arm")
CONTROL_HZ = 100.0
MAX_PRESSURE_PSI = 9.0

sim = make_sim(control_hz=CONTROL_HZ, seed=0)
obs = sim.reset()
print(f"Observations: {obs}")
print(f"soft-robotic-arm version: {installed_version}")
print("Simulator ready")
print("Command rows: S1, S2, S3, S4; columns: P1, P2, P3, P4, P5")
print("Tip position [m]:", np.round(obs["tip_pos"], 4))
print("Pouch-pressure array shape:", obs["pouch_pressures"].shape)


### What the simulator setup code does

- Selects EGL so MuJoCo can render without a physical display in Colab.
- Clears any older `soft_robotic_arm` import left in the current runtime.
- Imports NumPy, Matplotlib, ImageIO, and the `make_sim` simulator factory.
- Reports the installed package version.
- Creates the arm at a 100 Hz control rate; `seed=0` makes sensor noise repeatable.
- Calls `sim.reset()` to place the arm in its initial state and return the first
  sensor observation in `obs`.

## 3 — Command every pouch and read its pressure

Use a `(4, 5)` array to command the twenty pouches independently:

```python
command[segment_index, pouch_index] = pressure_psi
obs = sim.step(command)
```

Rows are S1–S4; columns are P1–P5. `sim.step()` also accepts a `(4,)` segment
vector and broadcasts each segment value to its five pouches. Keep every
command between 0 and 9 psi.

| Observation | Shape | Meaning |
|---|---:|---|
| `obs["tip_pos"]` | `(3,)` | tip `[x, y, z]` position in metres |
| `obs["tip_vel"]` | `(3,)` | tip velocity in m/s |
| `obs["pouch_pressures"]` | `(4, 5)` | measured pressure of every pouch |
| `obs["segment_pressures"]` | `(4,)` | mean measured pressure of each segment |
| `obs["time"]` | scalar | simulation time in seconds |


In [ ]:
# Commands are absolute pressures in psi. Rows select S1--S4 and
# columns select P1--P5, so the complete command has shape (4, 5).
command = np.full((4, 5), 2.0)  # Hold every pouch at a 2 psi baseline.
command[1, 2] = 6.0             # Raise only S2, P3 to 6 psi.

# One call to step() advances one control interval (1 / CONTROL_HZ seconds).
# Repeating it CONTROL_HZ times therefore applies this command for one second.
for _ in range(int(CONTROL_HZ)):
    obs = sim.step(command)

# The commanded and measured values differ because the plant includes pneumatic
# lag, deformation-dependent sensing, and sensor noise.
print("Commanded pouch pressures [psi]:\n", command)
print("Measured pouch pressures [psi]:\n", np.round(obs["pouch_pressures"], 3))
print("Measured segment means [psi]:", np.round(obs["segment_pressures"], 3))
print("Tip position [m]:", np.round(obs["tip_pos"], 4))


### Try any pouch

Choose `TEST_SEGMENT` from 0–3 and `TEST_POUCH` from 0–4. The cell holds every
other pouch at 2 psi, raises only the selected pouch, and plots its response.


In [ ]:
# Change these two indices to investigate any of the twenty pouches.
TEST_SEGMENT = 0  # 0=S1, 1=S2, 2=S3, 3=S4
TEST_POUCH = 0    # 0=P1, 1=P2, 2=P3, 3=P4, 4=P5

# Use a fresh simulator so this experiment does not inherit the previous state.
sim = make_sim(control_hz=CONTROL_HZ, seed=1)
obs = sim.reset()
baseline = np.full((4, 5), 2.0)

# Let the pneumatic and mechanical states settle before measuring displacement.
for _ in range(int(3.0 * CONTROL_HZ)):
    obs = sim.step(baseline)
home = obs["tip_pos"].copy()  # Reference position after settling [m].

# Perturb one pouch while leaving the other nineteen at the baseline pressure.
command = baseline.copy()
command[TEST_SEGMENT, TEST_POUCH] = 6.0

# Record four seconds of tip motion and all twenty measured pressures.
times, tips, measured = [], [], []
for _ in range(int(4.0 * CONTROL_HZ)):
    obs = sim.step(command)
    times.append(obs["time"])
    tips.append(obs["tip_pos"].copy())
    measured.append(obs["pouch_pressures"].copy())

# Convert lists to arrays for vectorized analysis and plotting.
times = np.asarray(times) - times[0]  # Start this plot at t = 0 s.
tips = np.asarray(tips)               # Shape: (samples, xyz).
measured = np.asarray(measured)       # Shape: (samples, segment, pouch).

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Plot planar displacement relative to the settled position; convert m to mm.
axes[0].plot(times, (tips[:, 0] - home[0]) * 1000, label="Δx")
axes[0].plot(times, (tips[:, 1] - home[1]) * 1000, label="Δy")
axes[0].set(xlabel="Time [s]", ylabel="Tip change [mm]", title="Tip response")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Show all five pressure sensors in the selected segment.
for pouch in range(5):
    axes[1].plot(times, measured[:, TEST_SEGMENT, pouch], label=f"P{pouch + 1}")
axes[1].set(xlabel="Time [s]", ylabel="Measured pressure [psi]", title=f"S{TEST_SEGMENT + 1} pouch pressures")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4 — Cartesian PD control, allocated across 3 real segments

This example is a closed-loop task-space controller matched to the real
evaluation harness (`soft_robotic_arm.evaluation`). At each 100 Hz update it
uses the measured planar tip position and velocity, a reference supplied by
the harness, and follows this architecture:

`circular reference (from the harness) → Cartesian PD → segment pressure deltas → pseudo-inverse allocation across S2–S4 → arm physics → tip feedback`

Unlike the earlier version, the controller no longer invents its own
reference trajectory. `CircularTrackingTask` / `evaluate_controller()` (used
in Section 5) generate a fixed 7 mm-radius, 0.1 Hz circle and pass the target
tip position in directly as `ref` on every call to `compute(t, obs, ref)`.

The feedback law still produces a planar **pressure correction**:

$\Delta\mathbf p_{xy}=K_p(\mathbf r_d-\mathbf r)-K_d\mathbf v$

$K_p$ has units psi/m and $K_d$ has units psi·s/m, same as before. The one
change: the harness doesn't supply a reference velocity, so the D term damps
the *measured* tip velocity directly (derivative-on-measurement) instead of
tracking a known target velocity.

The real plant only exposes **3 commandable segments** — S2, S3, S4 at 90°,
180°, 270° azimuth — because Segment 1 (0°, East) is a sealed reservoir on
the calibrated/real arm. With only 3 actuators, there is no clean symmetric
opposing pair (the old "+Δp_x to S1, −Δp_x to S3" trick needed 4 segments).
Instead, each segment's bending-contribution direction is built from its
azimuth, $\text{col\_axis}=(-\sin\phi,\cos\phi)$, stacked into a $(3,2)$
matrix, and the Moore–Penrose pseudo-inverse of that matrix maps the desired
$(\Delta p_x,\Delta p_y)$ onto the 3 actual segment deltas with minimum-norm
commands:

$\Delta\mathbf p_{\text{seg}}=\mathbf A^{+}\Delta\mathbf p_{xy},\qquad \mathbf A^{+}=\operatorname{pinv}(\mathbf A),\ \mathbf A\in\mathbb R^{3\times2}$

Each segment then gets exactly **one** pressure number (no five-pouch
`pouch_profile` split anymore — the real/calibrated plant commands a single
pressure per segment, not five independent pouches):

$p_{s}=\operatorname{clip}(p_0+\Delta p_{\text{seg},s},\,0,\,p_{max})$

A symmetric 4.5 psi baseline still provides room for both signs without a
lateral preference. `K_p` sets correction strength, `K_d` adds damping and
reduces phase lag — tune these first, same as before. Saturation, the
pneumatic lag, the real plant's extra 0.5 s transport delay and per-segment
regulator miscalibration, and nonlinear arm mechanics all limit performance,
so this remains a defensible baseline controller rather than an
inverse-dynamics controller.


In [ ]:
class RandomTrajectoryPDController:
    """Cartesian tip-space PD controller, adapted to the REAL evaluation harness.

    CHANGED (to match soft_robotic_arm.evaluation.Controller protocol):
      - compute(t, obs, ref) now takes the reference directly as a third
        argument, instead of generating one internally via self.reference().
        The real harness (CircularTrackingTask / evaluate_controller) owns
        trajectory generation now, not the controller.
      - Returns a (3,) array for Segments 2, 3, 4 only, not a (4, 5) pouch
        matrix. Segment 1 is a sealed reservoir on the real/calibrated plant
        and cannot be commanded directly, and there is no per-pouch
        allocation anymore -- one pressure number per segment, not five.
      - The old symmetric "S1/S3 = x pair, S2/S4 = y pair" opposing-segment
        trick is gone. With only 3 actuators at 90/180/270 degrees (no 0 deg
        segment), there's no clean symmetric pair, so the (ux, uy) PD
        correction is allocated across the 3 segments with a least-squares
        pseudo-inverse instead.
      - __init__ no longer builds its own random reference trajectory
        (dropped span_m, trajectory_seed, frequencies/amplitudes/phases) --
        the harness supplies ref every step, so that machinery is unused now.
    """

    def __init__(self, kp=500.0, kd=20.0, baseline_psi=4.5):
        # Cartesian feedback gains -- same role as the original. Tune these
        # first when studying performance.
        self.kp = kp  # Position gain [psi/m].
        self.kd = kd  # Velocity gain [psi*s/m].
        self.baseline_psi = baseline_psi

        # CHANGED: build an allocation matrix for the 3 real segments instead
        # of the old 4-segment opposing-pair logic. Segments 2, 3, 4 sit at
        # azimuths 90, 180, 270 deg (Segment 1 / 0 deg is sealed on the real
        # plant). Each segment's bending contribution direction follows the
        # simulator convention col_axis = (-sin(phi), cos(phi)).
        azimuths_deg = np.array([90.0, 180.0, 270.0])
        phis = np.deg2rad(azimuths_deg)
        col_axes = np.stack([-np.sin(phis), np.cos(phis)], axis=1)  # (3, 2)
        # Pseudo-inverse maps a desired planar (ux, uy) pressure-delta onto
        # the 3 actual segment pressure deltas with minimum-norm commands.
        self.alloc = np.linalg.pinv(col_axes.T)  # (3, 2): use as alloc @ [ux, uy]

    def compute(self, t: float, obs: dict, ref: np.ndarray) -> np.ndarray:
        """Compute one (3,) pressure command for Segments 2, 3, 4 from tip feedback.

        CHANGED: takes `ref` (the desired [x, y, z] tip position for this
        step) as an argument instead of calling self.reference(t). Also no
        longer tracks self.center_xy -- the harness centres the circle itself
        during its settle phase.
        """
        # Outer-loop task-space PD feedback on x, y only -- same idea as the
        # original.
        target_xy = np.asarray(ref[:2], dtype=float)
        position_error = target_xy - obs["tip_pos"][:2]
        # CHANGED: the harness doesn't hand us a reference velocity, so the D
        # term just damps actual tip velocity (derivative-on-measurement)
        # instead of tracking a known target velocity like before.
        velocity_error = -obs["tip_vel"][:2]
        xy_pressure_correction = self.kp * position_error + self.kd * velocity_error

        # CHANGED: allocate the 2D (ux, uy) correction across the 3 real
        # segments via the pseudo-inverse built in __init__, instead of the
        # old explicit S1/S3, S2/S4 opposing-pair formula.
        segment_pressure_delta = self.alloc @ xy_pressure_correction  # (3,)

        # CHANGED: one command per segment (no five-pouch profile -- the real
        # plant takes a single pressure per commandable segment).
        command = self.baseline_psi + segment_pressure_delta

        # Pressure saturation -- same idea as original, same 0-9 psi limit.
        return np.clip(command, 0.0, MAX_PRESSURE_PSI)


## 5 — Track the real circular reference via the evaluation harness

This section replaces the old hand-rolled loop with the actual grading
harness: `CircularTrackingTask` defines a 7 mm-radius, 0.1 Hz circle (5 s
settle at the baseline command, then 30 s tracked), and `evaluate_controller()`
runs the full settle-and-track loop against `make_calibrated_sim()` — the
plant with the extra 0.5 s transport delay and per-segment regulator
miscalibration that the real/graded plant has, on top of the same pneumatic
lag used everywhere else in this notebook.

The controller must return a finite `(3,)` array (Segments 2, 3, 4 pressures
in psi) on every call to `compute(t, obs, ref)`; `evaluate_controller()`
enforces that contract itself. Because the task and plant are both fixed and
seeded, results stay directly comparable across gain choices. It reports four
numbers: `rmse_mm` and `max_error_mm` (planar tracking accuracy),
`phase_lag_s` (cross-correlation lag between the reference and the tip, a
direct symptom of the pneumatic + transport delay), and `effort_psi` (RMS
command deviation from the 4.5 psi baseline). `result.plot()` draws the tip
path, lateral error, and S2/S3/S4 commands in one figure.


In [ ]:
# CHANGED: this entire cell now uses the real evaluation harness instead of
# a hand-rolled loop. make_sim() -> make_calibrated_sim() (the plant with the
# extra 0.5s transport delay + per-segment regulator miscalibration that the
# real/graded plant has), and the manual for-loop + RMSE math is replaced by
# evaluate_controller(), which owns settling, reference generation, stepping,
# and all four metrics (rmse_mm, max_error_mm, phase_lag_s, effort_psi).
task = CircularTrackingTask()  # 7mm radius, 0.1Hz, 5s settle + 30s tracked, 4.5psi baseline on S2-S4
sim = make_calibrated_sim(control_hz=CONTROL_HZ, seed=0)
controller = RandomTrajectoryPDController()

result = evaluate_controller(controller, task=task, sim=sim)

print(f"Task: {task.radius_m*1000:.0f}mm radius circle @ {task.frequency_hz}Hz, "
      f"{task.track_s:.0f}s tracked after {task.settle_s:.0f}s settle")
print(f"RMSE:        {result.rmse_mm:.2f} mm")
print(f"Max error:   {result.max_error_mm:.2f} mm")
print(f"Phase lag:   {result.phase_lag_s:.3f} s")
print(f"Effort:      {result.effort_psi:.3f} psi (RMS deviation from baseline)")

# CHANGED: result.plot() replaces the hand-built 6-panel figure (which plotted
# 4 segments x 5 pouches) -- the real controller only commands 3 segments, so
# the harness's own plot (tip path, lateral error, S2/S3/S4 commands) is the
# right one to use now.
fig, axes = result.plot(label="PD controller (real harness)")
plt.show()


## 6 — Render the controller simulation video (Optional)

This section runs the same controller in a fresh `make_calibrated_sim()` —
matching the plant actually scored in Section 5 — solely to render its
motion. It does not calculate metrics or create plots. Because the
controller no longer generates its own reference, this cell builds the same
circular reference with `make_reference()` on every step, the same way
`evaluate_controller()` does internally. Change the gains in
`RandomTrajectoryPDController`, then rerun this cell to generate a new
`soft_arm_pd_tracking.mp4`.


In [ ]:
# CHANGED: updated this whole cell so it still runs after the Cell 10/12
# changes above. The controller's compute() now needs (t, obs, ref) and
# returns a (3,) command, not (t, obs) -> (4, 5), so the video loop has to
# build its own reference each step (make_reference) the same way
# evaluate_controller() does internally, and switched to make_calibrated_sim()
# so the video matches the plant actually being scored in Cell 12.
VIDEO_RUN_TIME_S = 20.0
VIDEO_FPS = 10
VIDEO_PATH = "soft_arm_pd_tracking.mp4"
VIDEO_STRIDE = max(1, int(round(CONTROL_HZ / VIDEO_FPS)))

# Use a fresh simulation so video rendering is independent of the plotting run.
video_task = CircularTrackingTask()  # CHANGED: added -- same task settings as Cell 12.
video_sim = make_calibrated_sim(control_hz=CONTROL_HZ, seed=0)  # CHANGED: was make_sim(...)
video_controller = RandomTrajectoryPDController()
video_obs = video_sim.reset()

# CHANGED: added -- settle at the baseline command first, same as
# evaluate_controller() does, so the recorded video starts from the same
# steady circle centre instead of the raw reset transient.
video_baseline_command = video_task.baseline_command(MAX_PRESSURE_PSI)
for _ in range(video_task.settle_steps):
    video_obs = video_sim.step(video_baseline_command)
video_home = video_obs["tip_pos"].copy()

video_writer = imageio.get_writer(
    VIDEO_PATH, fps=VIDEO_FPS, codec="libx264", quality=7
)
print(f"Running {VIDEO_RUN_TIME_S:.0f} s simulation and recording {VIDEO_PATH} ...")

for step_index in range(int(VIDEO_RUN_TIME_S * CONTROL_HZ)):
    # CHANGED: build the reference explicitly with make_reference() and pass
    # it into compute() as the third argument -- the controller no longer
    # generates its own trajectory internally.
    video_t = step_index / CONTROL_HZ
    video_ref = make_reference(video_home, video_t, video_task)
    video_command = np.asarray(
        video_controller.compute(video_obs["time"], video_obs, video_ref), dtype=float
    )
    # CHANGED: contract check updated to (3,) -- Segments 2, 3, 4 only.
    if video_command.shape != (3,) or not np.all(np.isfinite(video_command)):
        video_writer.close()
        video_sim.close()
        raise ValueError("Controller must return a finite (3,) pressure command for Segments 2-4")

    video_obs = video_sim.step(video_command)

    # Render at 10 FPS while the controller itself continues to run at 100 Hz.
    if step_index % VIDEO_STRIDE == 0:
        frame = video_sim.render_frame(width=640, height=480)
        video_writer.append_data(frame)

video_writer.close()
print(f"Video saved to {VIDEO_PATH}")
display(Video(VIDEO_PATH, embed=True, width=800, html_attributes="controls loop"))
video_sim.close()  # Release MuJoCo's off-screen renderer.


## 7 — A reduced mechanical model, for sliding-mode control

Sections 4-6 used a **model-free** controller: PD only reacts to tip error,
it never uses the arm's mass, stiffness, or damping. Sliding-mode control
(SMC) needs an explicit model to compute, so this section derives one
directly from the real MuJoCo arm (not a guess, not a separate hand-fit) and
reduces it down to something small enough to control.

**Why reduction is needed.** The arm has 5 pouch levels x 3 bending/axial DOF
= 15 joint-space degrees of freedom. Working with all 15 is unnecessary: a
generalized eigenvalue decomposition of the bending-chain mass matrix
$M$ against its (uniform, diagonal) stiffness matrix $K$ shows **one
dominant vibration mode** sits at roughly 0.5 Hz and bends every level the
same direction (a single "C-curve" of the arm). The next mode is already at
~3 Hz and is 40x more heavily damped. Since the tracked circle here is only
0.1 Hz, everything above mode 0 is irrelevant to this controller and is
dropped. `x` and `y` bending are decoupled (near-zero coupling in $M$) and,
by the arm's symmetry, share the same reduced constants.

**The reduced model.** Projecting mode 0 onto tip position gives a single
second-order ODE per axis:

$$\ddot y + c\,\dot y + k\,y = b\,\Delta p_y$$

where $y$ is tip position along that axis (m) and $\Delta p_y$ is the same
net "y-direction pressure" quantity the Section 4 allocator already works
with (psi). $k$, $c$, $b$ are computed below directly from
`ArmConfig`/`build_arm_xml()` — nothing here is hand-tuned.

One finding worth flagging: this mechanical mode turns out to be
**overdamped** ($\zeta>1$), i.e. two real poles, not a ringing spring. The
slower pole's time constant is close to **0.7 s** — comparable to the 0.6 s
pneumatic lag and 0.5 s transport delay already in the plant. That means a
good chunk of the tracking lag PD could never beat (Section 5/6) is the
silicone itself being slow to unbend, not only the actuators.


In [ ]:
# NEW: derive k, c, b for the reduced single-axis model directly from the
# installed arm, by (1) building the real MuJoCo model, (2) extracting the
# joint-space mass matrix M(q=0) and the uniform bending stiffness/damping,
# (3) solving the generalized eigenvalue problem on the by0..by4 sub-chain,
# and (4) picking the mode with the largest tip authority. This mirrors the
# bx chain exactly (same stiffness/damping/geometry), so the same k, c, b
# are reused for both x and y below.
import mujoco
from scipy.linalg import eigh
from soft_robotic_arm.model import ArmConfig, build_arm_xml

_cfg = ArmConfig()
_xml = build_arm_xml(_cfg)
_model = mujoco.MjModel.from_xml_string(_xml)
_data = mujoco.MjData(_model)
mujoco.mj_forward(_model, _data)

_nv = _model.nv
_M = np.zeros((_nv, _nv))
mujoco.mj_fullM(_model, _data, _M)

_names = [mujoco.mj_id2name(_model, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(_model.njnt)]
_dofadr = {n: _model.jnt_dofadr[i] for i, n in enumerate(_names)}
_by_idx = [_dofadr[f"by{k}"] for k in range(5)]

_M_sub = _M[np.ix_(_by_idx, _by_idx)]
_K_sub = np.eye(5) * _cfg.base_stiffness
_D_sub = np.eye(5) * _cfg.base_damping

# Lever arms d(tip_x)/d(theta_by_k), by finite difference on the real model.
_tip_adr = _model.sensor_adr[mujoco.mj_name2id(_model, mujoco.mjtObj.mjOBJ_SENSOR, "tip_pos")]
_base_tip = _data.sensordata[_tip_adr:_tip_adr + 3].copy()
_eps = 0.01
_L = np.zeros(5)
for _k in range(5):
    _data.qpos[:] = 0.0
    _data.qpos[_by_idx[_k]] = _eps
    mujoco.mj_forward(_model, _data)
    _tip = _data.sensordata[_tip_adr:_tip_adr + 3].copy()
    _L[_k] = (_tip[0] - _base_tip[0]) / _eps
_data.qpos[:] = 0.0
mujoco.mj_forward(_model, _data)

_evals, _evecs = eigh(_K_sub, _M_sub)  # M-normalized eigenvectors: v^T M v = 1
_omegas = np.sqrt(np.clip(_evals, 0.0, None))
_force_gain = _cfg.pressure_gain * _cfg.moment_arm  # Nm per psi, per level

# Pick the mode with the largest static (steady-state) tip gain per psi,
# |tip_gain * force_factor / omega^2| -- NOT just the largest force-coupling
# numerator. At steady state (y''=0, y'=0) the model gives y = (b/k)*dp, so
# dividing by omega^2 (=k) is required to find which mode actually dominates
# the tip's *slow* response -- relevant since the tracked circle is 0.1 Hz,
# far below every mode's own frequency. Skipping the omega^2 division would
# instead pick whichever mode couples hardest to force, which can be a
# stiffer, practically-irrelevant higher mode.
_best_r, _best_score = 0, -1.0
for _r in range(5):
    _v = _evecs[:, _r]
    _tip_gain = _L @ _v
    _force_factor = _v @ np.ones(5)
    _static_gain = abs(_tip_gain * _force_factor) / max(_omegas[_r] ** 2, 1e-9)
    if _static_gain > _best_score:
        _best_r, _best_score = _r, _static_gain

_v = _evecs[:, _best_r]
smc_omega0 = _omegas[_best_r]                 # rad/s
smc_k = smc_omega0 ** 2                        # 1/s^2   (y'' coefficient on y)
smc_c = float(_v @ _D_sub @ _v)                # 1/s     (y'' coefficient on y')
_tip_gain = float(_L @ _v)                     # m/rad
_force_factor = float(_v @ np.ones(5))         # unitless
smc_b = _tip_gain * _force_gain * _force_factor  # m per psi

print(f"Dominant mode: {smc_omega0/(2*np.pi):.3f} Hz  (mode index {_best_r} of 0-4)")
print(f"Reduced model:  y'' + {smc_c:.4f} y' + {smc_k:.4f} y = {smc_b:.5f} * dp")
_zeta = smc_c / (2 * smc_omega0)
print(f"Damping ratio zeta = {_zeta:.3f}  ({'overdamped' if _zeta > 1 else 'underdamped'})")
if _zeta > 1:
    _s1 = (-smc_c + np.sqrt(smc_c**2 - 4*smc_k)) / 2
    _s2 = (-smc_c - np.sqrt(smc_c**2 - 4*smc_k)) / 2
    print(f"Real poles: {_s1:.4f}, {_s2:.4f}  ->  time constants {-1/_s1:.3f}s, {-1/_s2:.3f}s")


## 8 — Sliding-mode controller on the reduced model

With $k$, $c$, $b$ from Section 7, the per-axis plant is
$\ddot y = -k y - c \dot y + b\,\Delta p$. SMC is built the standard way:

**Error and sliding surface.** With $e=y-y_d$:

$$s = \dot e + \lambda e$$

Driving $s\to 0$ makes the tracking error decay like $\dot e=-\lambda e$ —
first-order, no overshoot, rate set by $\lambda$.

**Equivalent control + switching term.** Differentiate $s$, substitute the
plant equation, and force $\dot s=-K\,\mathrm{sat}(s/\phi)$ (a boundary-layer
smoothed sliding condition — avoids chattering the real pressure regulators):

$$\Delta p = \frac{1}{b}\Big[k\,y + c\,\dot y + \ddot y_d - \lambda\,\dot e - K\,\mathrm{sat}(s/\phi)\Big]$$

The first three terms inside the brackets are the **equivalent control**
(model-based feedforward/feedback that would hold $s=0$ exactly if the
reduced model were perfect); $-K\,\mathrm{sat}(s/\phi)$ is the **robust
term** that pulls the trajectory back onto the surface despite whatever the
reduced model gets wrong (actuator lag, delay, regulator miscalibration,
the 4 discarded higher modes).

**Reference derivatives.** The harness only hands `compute()` a position
`ref` each step, not $\dot y_d$/$\ddot y_d$. Since the circular reference is
smooth, the controller estimates both itself with simple backward finite
differences over the last 3 calls — no knowledge of `CircularTrackingTask`'s
internals is used, so this still only consumes the `(t, obs, ref)` contract.

**Allocation.** $\Delta p_x,\Delta p_y$ from the two independent (x, y)
sliding-mode laws plug into the *same* pseudo-inverse allocator built in
Section 4 ($\Delta\mathbf p_{\text{seg}}=\mathbf A^{+}\Delta\mathbf p_{xy}$)
to produce the 3 segment pressures, then baseline + clip as before.

**What this does and doesn't fix.** This controller is still driving
*through* the 0.6 s pneumatic lag and 0.5 s transport delay and the
per-segment regulator miscalibration — none of that is modeled here, it all
lands on the robust term $-K\,\mathrm{sat}(s/\phi)$. SMC's real advantage
over PD is robustness to exactly those unmodeled errors, not delay
compensation. Don't expect the phase lag to disappear; a disturbance
observer that explicitly predicts 0.5 s ahead is the next step for that
(future work, not implemented here).


In [ ]:
# NEW: sliding-mode controller using the Section 7 reduced model, matching
# the same Controller protocol as RandomTrajectoryPDController in Section 4
# (compute(t, obs, ref) -> (3,) array for Segments 2, 3, 4). Does NOT modify
# or reuse state from RandomTrajectoryPDController -- fully independent class.
class SlidingModeController:
    """Cartesian-space SMC on the reduced tip model from Section 7.

    lam     : sliding-surface slope (1/s) -- how fast tracking error decays.
    K_switch: robust/switching gain (psi) -- how hard it fights unmodeled
              effects (actuator lag+delay, regulator miscalibration, the
              higher modes dropped in the reduction).
    phi     : boundary-layer half-width (m/s, same units as s) -- larger
              phi = smoother command but slower/less precise sliding;
              smaller phi = tighter tracking but more chatter.
    """

    def __init__(self, k=smc_k, c=smc_c, b=smc_b, lam=5.0, K_switch=0.2, phi=0.02,
                 baseline_psi=4.5):
        self.k, self.c, self.b = k, c, b
        self.lam = lam
        self.K_switch = K_switch
        self.phi = phi
        self.baseline_psi = baseline_psi

        # Same 3-segment pseudo-inverse allocator as Section 4's PD controller.
        azimuths_deg = np.array([90.0, 180.0, 270.0])
        phis = np.deg2rad(azimuths_deg)
        col_axes = np.stack([-np.sin(phis), np.cos(phis)], axis=1)
        self.alloc = np.linalg.pinv(col_axes.T)

        # History for finite-difference reference derivatives.
        self._t_hist = []
        self._ref_hist = []

    def _ref_derivatives(self, t, ref_xy):
        """Backward-difference estimate of ref velocity/acceleration from the
        last up-to-3 calls. Returns zeros until enough history is buffered
        (only the first couple of steps of the tracked phase)."""
        self._t_hist.append(t)
        self._ref_hist.append(ref_xy.copy())
        if len(self._t_hist) > 3:
            self._t_hist.pop(0)
            self._ref_hist.pop(0)

        if len(self._t_hist) < 2:
            return np.zeros(2), np.zeros(2)

        dt1 = self._t_hist[-1] - self._t_hist[-2]
        ref_vel = (self._ref_hist[-1] - self._ref_hist[-2]) / dt1

        if len(self._t_hist) < 3:
            return ref_vel, np.zeros(2)

        dt0 = self._t_hist[-2] - self._t_hist[-3]
        ref_vel_prev = (self._ref_hist[-2] - self._ref_hist[-3]) / dt0
        ref_acc = (ref_vel - ref_vel_prev) / dt1
        return ref_vel, ref_acc

    def compute(self, t: float, obs: dict, ref: np.ndarray) -> np.ndarray:
        target_xy = np.asarray(ref[:2], dtype=float)
        ref_vel, ref_acc = self._ref_derivatives(t, target_xy)

        y = obs["tip_pos"][:2]
        y_dot = obs["tip_vel"][:2]

        e = y - target_xy
        e_dot = y_dot - ref_vel
        s = e_dot + self.lam * e

        # Boundary-layer saturation instead of sign(s) -- continuous command,
        # no regulator-shredding chatter.
        sat_s = np.clip(s / self.phi, -1.0, 1.0)

        # Equivalent control (model-based) + robust switching term, solved
        # from  y_ddot = -k*y - c*y_dot + b*dp  with  s_dot = -K*sat(s/phi).
        dp_xy = (self.k * y + self.c * y_dot + ref_acc - self.lam * e_dot
                 - self.K_switch * sat_s) / self.b

        segment_pressure_delta = self.alloc @ dp_xy
        command = self.baseline_psi + segment_pressure_delta
        return np.clip(command, 0.0, MAX_PRESSURE_PSI)


## 9 — Evaluate the SMC controller on the real harness

Exactly the same `CircularTrackingTask` / `make_calibrated_sim()` /
`evaluate_controller()` used for the PD baseline in Section 5, so the
numbers below are directly comparable to the PD results you already
recorded. The defaults below (`lam=5.0, K_switch=0.2, phi=0.02`) were found
by a short gain sweep and already beat the best PD run (~4.4 mm RMSE /
0.8 s lag vs PD's ~9.9 mm / 2.1 s) — tune further the same empirical way you
tuned `kp`/`kd`, one parameter at a time.


In [ ]:
smc_task = CircularTrackingTask()
smc_sim = make_calibrated_sim(control_hz=CONTROL_HZ, seed=0)
smc_controller = SlidingModeController(lam=5.0, K_switch=0.2, phi=0.02)

smc_result = evaluate_controller(smc_controller, task=smc_task, sim=smc_sim)

print(f"Task: {smc_task.radius_m*1000:.0f}mm radius circle @ {smc_task.frequency_hz}Hz, "
      f"{smc_task.track_s:.0f}s tracked after {smc_task.settle_s:.0f}s settle")
print(f"RMSE:        {smc_result.rmse_mm:.2f} mm")
print(f"Max error:   {smc_result.max_error_mm:.2f} mm")
print(f"Phase lag:   {smc_result.phase_lag_s:.3f} s")
print(f"Effort:      {smc_result.effort_psi:.3f} psi (RMS deviation from baseline)")

fig, axes = smc_result.plot(label="SMC controller (reduced model)")
plt.show()


## 10 — Compare against the PD baseline

Runs `RandomTrajectoryPDController` from Section 4 (unchanged) back-to-back
with the SMC controller above, on the identical task/plant/seed, so the
side-by-side numbers isolate the controller's contribution only.


In [ ]:
pd_task = CircularTrackingTask()
pd_sim = make_calibrated_sim(control_hz=CONTROL_HZ, seed=0)
pd_controller = RandomTrajectoryPDController()  # Section 4 defaults; edit kp/kd to match your tuning
pd_result = evaluate_controller(pd_controller, task=pd_task, sim=pd_sim)

print(f"{'Controller':<12}{'RMSE (mm)':>12}{'Max err (mm)':>14}{'Phase lag (s)':>16}{'Effort (psi)':>14}")
print(f"{'PD':<12}{pd_result.rmse_mm:>12.2f}{pd_result.max_error_mm:>14.2f}"
      f"{pd_result.phase_lag_s:>16.3f}{pd_result.effort_psi:>14.3f}")
print(f"{'SMC':<12}{smc_result.rmse_mm:>12.2f}{smc_result.max_error_mm:>14.2f}"
      f"{smc_result.phase_lag_s:>16.3f}{smc_result.effort_psi:>14.3f}")


## 11 — Render the SMC controller simulation video (Optional)

Same pattern as Section 6's PD video, but driving `SlidingModeController`
instead, on a fresh `make_calibrated_sim()` run so the clip matches the
plant actually scored in Section 9. Re-run this cell after changing
`lam`/`K_switch`/`phi` to see the new behavior.


In [ ]:
SMC_VIDEO_RUN_TIME_S = 20.0
SMC_VIDEO_FPS = 10
SMC_VIDEO_PATH = "soft_arm_smc_tracking.mp4"
SMC_VIDEO_STRIDE = max(1, int(round(CONTROL_HZ / SMC_VIDEO_FPS)))

smc_video_task = CircularTrackingTask()
smc_video_sim = make_calibrated_sim(control_hz=CONTROL_HZ, seed=0)
smc_video_controller = SlidingModeController()
smc_video_obs = smc_video_sim.reset()

# Settle at the baseline command first, same as evaluate_controller() does,
# so the recorded video starts from the steady circle centre.
smc_video_baseline_command = smc_video_task.baseline_command(MAX_PRESSURE_PSI)
for _ in range(smc_video_task.settle_steps):
    smc_video_obs = smc_video_sim.step(smc_video_baseline_command)
smc_video_home = smc_video_obs["tip_pos"].copy()

smc_video_writer = imageio.get_writer(
    SMC_VIDEO_PATH, fps=SMC_VIDEO_FPS, codec="libx264", quality=7
)
print(f"Running {SMC_VIDEO_RUN_TIME_S:.0f} s simulation and recording {SMC_VIDEO_PATH} ...")

for step_index in range(int(SMC_VIDEO_RUN_TIME_S * CONTROL_HZ)):
    smc_video_t = step_index / CONTROL_HZ
    smc_video_ref = make_reference(smc_video_home, smc_video_t, smc_video_task)
    smc_video_command = np.asarray(
        smc_video_controller.compute(smc_video_obs["time"], smc_video_obs, smc_video_ref),
        dtype=float,
    )
    if smc_video_command.shape != (3,) or not np.all(np.isfinite(smc_video_command)):
        smc_video_writer.close()
        smc_video_sim.close()
        raise ValueError("Controller must return a finite (3,) pressure command for Segments 2-4")

    smc_video_obs = smc_video_sim.step(smc_video_command)

    if step_index % SMC_VIDEO_STRIDE == 0:
        frame = smc_video_sim.render_frame(width=640, height=480)
        smc_video_writer.append_data(frame)

smc_video_writer.close()
print(f"Video saved to {SMC_VIDEO_PATH}")
display(Video(SMC_VIDEO_PATH, embed=True, width=800, html_attributes="controls loop"))
smc_video_sim.close()  # Release MuJoCo's off-screen renderer.
